In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings

from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

EDA_SAMPLE_N = 200_000

In [7]:
# Importing datafile 

DATA_PATH = "data/gaming_mental_health_10M_40features.csv"

USE_SAMPLE = True
NROWS = 200_000

data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(f"CSV not found at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, nrows=NROWS, low_memory=False) if USE_SAMPLE else pd.read_csv(DATA_PATH, low_memory=False)

print("df shape:", df.shape)
display(df.head())

mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
print(f"Approx. memory usage (loaded df): {mem_mb:.2f} MB")

df shape: (200000, 39)


,age,gender,income,daily_gaming_hours,weekly_sessions,years_gaming,sleep_hours,caffeine_intake,exercise_hours,stress_level,anxiety_score,depression_score,social_interaction_score,relationship_satisfaction,academic_performance,work_productivity,addiction_level,multiplayer_ratio,toxic_exposure,violent_games_ratio,mobile_gaming_ratio,night_gaming_ratio,weekend_gaming_hours,friends_gaming_count,online_friends,streaming_hours,esports_interest,headset_usage,microtransactions_spending,parental_supervision,loneliness_score,aggression_score,happiness_score,bmi,screen_time_total,eye_strain_score,back_pain_score,competitive_rank,internet_quality
0,51,Female,8615,3.68,22,17,5.26,1.00,0.18,3,4.06,5.02,7.85,9.55,68.80,77.12,2.02,0.39,0.05,0.51,0.23,0.32,1.50,20,186,2.82,1,1,1746.97,0,2.87,3.19,5.20,19.69,4.71,5.71,4.81,80,10
1,41,Female,39453,5.70,34,16,9.20,0.70,1.44,8,6.76,7.63,7.06,6.96,38.11,44.94,5.85,0.23,0.37,0.43,0.24,0.32,17.93,37,21,1.34,8,1,342.04,7,4.17,7.73,5.40,26.37,6.62,6.77,3.99,57,2
2,27,Male,40466,1.58,8,22,7.39,2.24,3.15,3,9.57,4.02,1.12,2.61,97.44,54.77,0.08,0.06,0.15,0.48,0.27,0.42,2.99,45,146,3.56,8,1,57.95,9,9.38,2.85,5.17,25.15,9.30,2.16,4.75,59,10
3,55,Male,51076,6.11,39,24,7.99,1.65,2.80,1,4.97,1.40,6.59,6.00,67.65,88.56,2.77,0.53,0.36,0.64,0.57,0.82,1.99,11,133,1.80,6,1,617.32,5,8.24,7.19,8.62,26.42,13.81,4.72,5.37,89,1
4,20,Male,86116,3.65,17,0,7.12,1.02,1.01,2,8.73,4.83,6.03,3.54,62.34,68.82,3.37,0.78,0.25,0.29,0.47,0.83,9.23,24,399,1.30,4,0,1398.39,9,6.65,2.53,9.71,25.75,10.74,3.90,6.44,15,10


Approx. memory usage (loaded df): 69.81 MB


In [8]:
# Setting up my features

DEPLOY_FEATURES = [
    "age",
    "gender",
    "daily_gaming_hours",
    "weekly_sessions",
    "years_gaming",
    "competitive_rank",
    "online_friends",
    "microtransactions_spending",
    "screen_time_total",
    "sleep_hours",
    "stress_level",
    "depression_score",
]

TARGET_SOURCE_COL = "addiction_level"
TARGET_COL = "high_addiction"
QUANTILE = 0.80

required = set(DEPLOY_FEATURES + [TARGET_SOURCE_COL])
missing = sorted(list(required - set(df.columns)))

print("Missing required columns:", missing)
if missing:
    print("\nAvailable columns:\n", sorted(df.columns.tolist()))
    raise ValueError("Dataset is missing required columns for the Final Project model.")

threshold = float(df[TARGET_SOURCE_COL].quantile(QUANTILE))
df[TARGET_COL] = (df[TARGET_SOURCE_COL] >= threshold).astype(int)

print(f"\n{QUANTILE:.0%} threshold for '{TARGET_SOURCE_COL}':", threshold)
print("Class balance (high_addiction=1 rate):", float(df[TARGET_COL].mean()))

X = df[DEPLOY_FEATURES].copy()
y = df[TARGET_COL].copy()

print("\nX shape:", X.shape, " | y shape:", y.shape)
display(X.head())
display(y.value_counts(dropna=False).to_frame("count").T)

Missing required columns: []

80% threshold for 'addiction_level': 4.44
Class balance (high_addiction=1 rate): 0.20089

X shape: (200000, 12)  | y shape: (200000,)


,age,gender,daily_gaming_hours,weekly_sessions,years_gaming,competitive_rank,online_friends,microtransactions_spending,screen_time_total,sleep_hours,stress_level,depression_score
0,51,Female,3.68,22,17,80,186,1746.97,4.71,5.26,3,5.02
1,41,Female,5.70,34,16,57,21,342.04,6.62,9.20,8,7.63
2,27,Male,1.58,8,22,59,146,57.95,9.30,7.39,3,4.02
3,55,Male,6.11,39,24,89,133,617.32,13.81,7.99,1,1.40
4,20,Male,3.65,17,0,15,399,1398.39,10.74,7.12,2,4.83


high_addiction,0,1
count,159822,40178


In [9]:
# Preprocessing the model

cat_cols = ["gender"]
num_cols = [c for c in DEPLOY_FEATURES if c not in cat_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

lr = LogisticRegression(max_iter=1000)

lr_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", lr)
])

print("Pipeline created:", lr_pipe)

Pipeline created: Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'daily_gaming_hours',
                                                   'weekly_sessions',
                                                   'years_gaming',
                                                   'competitive_rank',
                                                   'online_friends',
                                                   'microtransactions_spending',
                                                   'screen_time_total',
                                       

In [10]:
# Splitting Train, Validate, Test 70/15/15

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Split sizes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Train positive rate:", float(y_train.mean()))
print("Val positive rate:", float(y_val.mean()))
print("Test positive rate:", float(y_test.mean()))

Split sizes:
Train: (140000, 12) Val: (30000, 12) Test: (30000, 12)
Train positive rate: 0.20089285714285715
Val positive rate: 0.2009
Test positive rate: 0.20086666666666667


In [11]:
# Fit and Eval the LR Model

lr_pipe.fit(X_train, y_train)

val_proba = lr_pipe.predict_proba(X_val)[:, 1]
test_proba = lr_pipe.predict_proba(X_test)[:, 1]

val_pred = (val_proba >= 0.50).astype(int)
test_pred = (test_proba >= 0.50).astype(int)

val_auc = roc_auc_score(y_val, val_proba)
val_ap = average_precision_score(y_val, val_proba)

test_auc = roc_auc_score(y_test, test_proba)
test_ap = average_precision_score(y_test, test_proba)

print("\nValidation Metrics:")
print("ROC AUC:", round(val_auc, 4), " | Avg Precision:", round(val_ap, 4))
print("Precision:", round(precision_score(y_val, val_pred), 4),
      "Recall:", round(recall_score(y_val, val_pred), 4),
      "F1:", round(f1_score(y_val, val_pred), 4))

print("\nTest Metrics:")
print("ROC AUC:", round(test_auc, 4), " | Avg Precision:", round(test_ap, 4))
print("Precision:", round(precision_score(y_test, test_pred), 4),
      "Recall:", round(recall_score(y_test, test_pred), 4),
      "F1:", round(f1_score(y_test, test_pred), 4))

print("\nConfusion Matrix (Test):")
cm = confusion_matrix(y_test, test_pred)
display(pd.DataFrame(cm, index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"]))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))


Validation Metrics:
ROC AUC: 0.9639  | Avg Precision: 0.8915
Precision: 0.8362 Recall: 0.7435 F1: 0.7871

Test Metrics:
ROC AUC: 0.9661  | Avg Precision: 0.8986
Precision: 0.8441 Recall: 0.7537 F1: 0.7964

Confusion Matrix (Test):


,Pred 0,Pred 1
Actual 0,23135,839
Actual 1,1484,4542



Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9397    0.9650    0.9522     23974
           1     0.8441    0.7537    0.7964      6026

    accuracy                         0.9226     30000
   macro avg     0.8919    0.8594    0.8743     30000
weighted avg     0.9205    0.9226    0.9209     30000



In [12]:
# Saving Artifacts 

ARTIFACT_DIR = "artifacts_sm"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

model_out = os.path.join(ARTIFACT_DIR, "model.joblib")
joblib.dump(lr_pipe, model_out)

with open(os.path.join(ARTIFACT_DIR, "deploy_features.json"), "w") as f:
    json.dump(DEPLOY_FEATURES, f, indent=2)

with open(os.path.join(ARTIFACT_DIR, "high_addiction_threshold.txt"), "w") as f:
    f.write(str(float(threshold)))

bounds = {"features": {}}

bounds["features"]["gender"] = {"allowed": sorted([str(x) for x in pd.Series(X["gender"]).dropna().unique().tolist()])}

for c in [col for col in DEPLOY_FEATURES if col != "gender"]:
    s = pd.to_numeric(X[c], errors="coerce")
    bounds["features"][c] = {
        "min": float(np.nanmin(s.values)),
        "max": float(np.nanmax(s.values)),
    }

with open(os.path.join(ARTIFACT_DIR, "input_bounds.json"), "w") as f:
    json.dump(bounds, f, indent=2)

model_selection = {
    "selected_model": "logistic_regression",
    "pipeline": True,
    "decision_threshold": 0.50,
    "target_definition": f"{TARGET_COL} = 1 if {TARGET_SOURCE_COL} >= quantile({TARGET_QUANTILE}) else 0",
}
with open(os.path.join(ARTIFACT_DIR, "model_selection.json"), "w") as f:
    json.dump(model_selection, f, indent=2)

metrics = {}
for name in ["val_auc", "val_ap", "test_auc", "test_ap"]:
    if name in globals():
        metrics[name] = float(globals()[name])

with open(os.path.join(ARTIFACT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved artifacts to:", ARTIFACT_DIR)
print("Contents:", sorted(os.listdir(ARTIFACT_DIR)))
print("Model file:", model_out)

Saved artifacts to: artifacts_sm
Contents: ['deploy_features.json', 'high_addiction_threshold.txt', 'input_bounds.json', 'metrics.json', 'model.joblib', 'model_selection.json']
Model file: artifacts_sm/model.joblib


In [13]:
# Packaging my model and uploading to S3

import os, tarfile
import sagemaker

ARTIFACT_DIR = "artifacts_sm"
TAR_NAME = "model.tar.gz"

if not os.path.isdir(ARTIFACT_DIR):
    raise FileNotFoundError(f"Missing directory: {ARTIFACT_DIR}")

with tarfile.open(TAR_NAME, "w:gz") as tar:
    for fname in sorted(os.listdir(ARTIFACT_DIR)):
        fpath = os.path.join(ARTIFACT_DIR, fname)
        if os.path.isfile(fpath):
            tar.add(fpath, arcname=fname)

print("Created:", TAR_NAME)
with tarfile.open(TAR_NAME, "r:gz") as tar:
    print("Tar contents:", tar.getnames())

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "ana680-final/model"

model_s3_uri = sess.upload_data(TAR_NAME, bucket=bucket, key_prefix=prefix)
print("\nUploaded model.tar.gz to:", model_s3_uri)

Created: model.tar.gz
Tar contents: ['deploy_features.json', 'high_addiction_threshold.txt', 'input_bounds.json', 'metrics.json', 'model.joblib', 'model_selection.json']
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix

Uploaded model.tar.gz to: s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/ana680-final/model/model.tar.gz


In [15]:
# Creating my inference.py allows me to set up how my endpoint loads the model and handles prediction

import os
import boto3
from sagemaker import image_uris

model_s3_uri = "s3://amazon-sagemaker-357954864016-us-east-2-41b4kae9iij0o9/ana680-final/model/model.tar.gz"
print("model_s3_uri:", model_s3_uri)

inference_code = r'''
import json
import os
import joblib
import pandas as pd

DEPLOY_FEATURES = [
    "age",
    "gender",
    "daily_gaming_hours",
    "weekly_sessions",
    "years_gaming",
    "competitive_rank",
    "online_friends",
    "microtransactions_spending",
    "screen_time_total",
    "sleep_hours",
    "stress_level",
    "depression_score",
]

THRESHOLD = 0.50

def model_fn(model_dir: str):
    model_path = os.path.join(model_dir, "model.joblib")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Missing model file: {model_path}")
    return joblib.load(model_path)

def input_fn(request_body, request_content_type):
    if request_content_type != "application/json":
        raise ValueError(f"Unsupported content type: {request_content_type}")

    payload = json.loads(request_body)

    if isinstance(payload, dict) and "instances" in payload:
        instances = payload["instances"]
    else:
        instances = payload

    if isinstance(instances, dict):
        instances = [instances]

    if not isinstance(instances, list) or not all(isinstance(x, dict) for x in instances):
        raise ValueError("JSON must be a dict, a list of dicts, or {'instances': [..]}")

    df = pd.DataFrame(instances)

    # Ensure expected columns exist
    for c in DEPLOY_FEATURES:
        if c not in df.columns:
            df[c] = None

    return df[DEPLOY_FEATURES]

def predict_fn(input_data, model):
    proba = model.predict_proba(input_data)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    return {"probabilities": proba.tolist(), "predictions": pred.tolist(), "threshold": THRESHOLD}

def output_fn(prediction, accept):
    return json.dumps(prediction), "application/json"
'''.lstrip()

with open("inference.py", "w", encoding="utf-8") as f:
    f.write(inference_code)

print("Wrote inference.py:", os.path.abspath("inference.py"))

region = boto3.Session().region_name
candidate_versions = ["1.7-1", "1.6-1", "1.5-1", "1.4-1", "1.3-1", "1.2-1", "1.0-1", "0.23-1"]

chosen = None
for v in candidate_versions:
    try:
        _ = image_uris.retrieve("sklearn", region=region, version=v, py_version="py3", instance_type="ml.m5.large")
        chosen = v
        break
    except Exception:
        pass

print("Region:", region)
print("Chosen SageMaker sklearn framework_version:", chosen)

if chosen is None:
    raise RuntimeError("No SageMaker sklearn container version found from the candidate list.")

model_s3_uri: s3://amazon-sagemaker-357954864016-us-east-2-41b4kae9iij0o9/ana680-final/model/model.tar.gz
Wrote inference.py: /home/sagemaker-user/ANA680 Final/inference.py
Region: us-east-2
Chosen SageMaker sklearn framework_version: 1.2-1


In [19]:
# Uploading dataset to S3

import boto3
import sagemaker
import time
import os

LOCAL_DATA_PATH = "data/gaming_mental_health_10M_40features.csv"
if not os.path.exists(LOCAL_DATA_PATH):
    raise FileNotFoundError(f"Not found: {LOCAL_DATA_PATH}")

region = boto3.Session().region_name
sts = boto3.client("sts", region_name=region)
account_id = sts.get_caller_identity()["Account"]

suffix = str(int(time.time()))
new_bucket = f"ana680-{account_id}-{region}-{suffix}".lower()

s3 = boto3.client("s3", region_name=region)

print("Region:", region)
print("Creating/using bucket:", new_bucket)

try:
    s3.create_bucket(
        Bucket=new_bucket,
        CreateBucketConfiguration={"LocationConstraint": region},
    )
    print("Bucket created.")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print("Bucket already exists and is owned by you.")
except Exception as e:
    raise RuntimeError(f"Failed to create bucket {new_bucket}: {e}")

sess = sagemaker.Session()
prefix = "ana680-final/data"
s3_data_uri = sess.upload_data(LOCAL_DATA_PATH, bucket=new_bucket, key_prefix=prefix)

print("\nUploaded training data to:", s3_data_uri)

from urllib.parse import urlparse
u = urlparse(s3_data_uri)
b = u.netloc
k = u.path.lstrip("/")

s3.head_bucket(Bucket=b)
s3.head_object(Bucket=b, Key=k)
print("Verified: bucket + object exist.")

Region: us-east-2
Creating/using bucket: ana680-357954864016-us-east-2-1772162945
Bucket created.
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix

Uploaded training data to: s3://ana680-357954864016-us-east-2-1772162945/ana680-final/data/gaming_mental_health_10M_40features.csv
Verified: bucket + object exist.


In [20]:
# Training Container train.py using sklearn 1.2-1

import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from urllib.parse import urlparse

s3_data_uri = "s3://ana680-357954864016-us-east-2-1772162945/ana680-final/data/gaming_mental_health_10M_40features.csv"

u = urlparse(s3_data_uri)
bucket = u.netloc
key = u.path.lstrip("/")
prefix_dir = "/".join(key.split("/")[:-1]) + "/"
s3_train_prefix = f"s3://{bucket}/{prefix_dir}"

print("Training input prefix:", s3_train_prefix)

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

output_path = f"s3://{bucket}/ana680-final/output"
print("Output path:", output_path)

FRAMEWORK_VERSION = "1.2-1"

est = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version=FRAMEWORK_VERSION,
    py_version="py3",
    sagemaker_session=sess,
    output_path=output_path,
    hyperparameters={"quantile": 0.80, "random-state": 42},
)

est.fit({"train": s3_train_prefix})

print("Training job name:", est.latest_training_job.name)
print("Model artifact S3 (est.model_data):", est.model_data)

Training input prefix: s3://ana680-357954864016-us-east-2-1772162945/ana680-final/data/
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Output path: s3://ana680-357954864016-us-east-2-1772162945/ana680-final/output
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-27 03:32:07 Starting - Starting the training job...
2026-02-27 03:32:30 Starting - Preparing the instances for training...
2026-02-27 03:32:54 Downloading - Downloading input data...
2026-02-27 03:33:23 Downloading - Downloading the training image......
2026-02-27 03:34:24 Training - 

In [22]:
# Checking Training Job

desc = est.latest_training_job.describe()
status = desc["TrainingJobStatus"]

print("TrainingJobName:", desc["TrainingJobName"])
print("Status:", status)

if status == "Failed":
    print("FailureReason:", desc.get("FailureReason"))

if status == "Completed":
    print("Model artifact S3 (est.model_data):", est.model_data)

TrainingJobName: sagemaker-scikit-learn-2026-02-27-03-32-06-334
Status: Completed
Model artifact S3 (est.model_data): s3://ana680-357954864016-us-east-2-1772162945/ana680-final/output/sagemaker-scikit-learn-2026-02-27-03-32-06-334/output/model.tar.gz


In [23]:
# Deploying my Endpoint for my container

import time
import boto3
import botocore
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

model_artifact_s3 = "s3://ana680-357954864016-us-east-2-1772162945/ana680-final/output/sagemaker-scikit-learn-2026-02-27-03-32-06-334/output/model.tar.gz"

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
sm = boto3.client("sagemaker")

endpoint_name_base = "final-project-endpoint"
endpoint_name = endpoint_name_base

try:
    sm.describe_endpoint(EndpointName=endpoint_name)
    endpoint_name = f"{endpoint_name_base}-{time.strftime('%Y%m%d-%H%M%S')}"
except botocore.exceptions.ClientError as e:
    if "Could not find" in str(e) or "ValidationException" in str(e):
        pass
    else:
        raise

print("Endpoint name:", endpoint_name)

model = SKLearnModel(
    model_data=model_artifact_s3,
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Deployed endpoint:", endpoint_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Endpoint name: final-project-endpoint
------!Deployed endpoint: final-project-endpoint


In [24]:
# Testing my Endpoint

import sagemaker
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sess = sagemaker.Session()

endpoint_name = "final-project-endpoint"

predictor = Predictor(endpoint_name=endpoint_name, sagemaker_session=sess)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

payload = {
    "instances": [
        {
            "age": 35,
            "gender": "Male",
            "daily_gaming_hours": 4.0,
            "weekly_sessions": 20,
            "years_gaming": 10,
            "competitive_rank": 5,
            "online_friends": 120,
            "microtransactions_spending": 25.0,
            "screen_time_total": 7.5,
            "sleep_hours": 6.5,
            "stress_level": 6,
            "depression_score": 4.2
        }
    ]
}

result = predictor.predict(payload)
result

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


{'probabilities': [0.05055146886160844], 'predictions': [0], 'threshold': 0.5}